In [44]:
from Bio import SeqIO

record = SeqIO.read("MN328061.1_BD_2019_strain.fasta", "fasta")
genome = record.seq

E_gene = genome[953:2495]   # Python is 0-based 936: 2421 (955:2496)
print(len(E_gene))


1542


In [45]:
E_protein = E_gene.translate()
print(E_protein)
print(len(E_protein))

IETLWKGFQEEAGLT*SWNMEAV*RRWQKINQHWILN**KQKRNIPPL*GSIA*RQS*PTQQQHLAAQHKENLA*MKNRTKDLSANTPW*IEDGEMGADYLEREVS*PVQCSHAKRTWKGKSCNQKTWSTPL**HLTQGKRMQSEMTQENTARKLK*HHRVPSQRQN*QAMALSRWNALRERDSTSMRWCCCKWKTRLGWCTGNGS*TCHYHGCPEQTYKDQIGYRRRHWSLSKIPMQRNRMLLF*DPKKGLCTQHLQGPRKSRCHQETYCSQDILNVG*EWTNYSSKECHIPCVQESSKL*RK*QKHNMGQ*LSEYNMKGTVLRARSLLK*WIWKKDMS*VA*SQSTQLSQKKTAQST*KQNPHSETVTSL*E*NRDN*SSAGLRKEALLAKCLRQQ*EERREWPF*VTQLGILDPWEECSHL*ERPSTKSLEQSMGLPLVGFHGL*KSS*ESSSHG*E*IHVVLHCLCH*Y*WES*HCIWELWCRPIVVAL*AGKTKN*NVAVGFLSQTTCT
514


In [18]:
from Bio import pairwise2
from Bio import SeqIO

ref = SeqIO.read("E_reference.fasta", "fasta").seq
bd  = SeqIO.read("E_PQ657766.fasta", "fasta").seq

alignments = pairwise2.align.globalxx(ref, bd)
best = alignments[0]

print(best.seqA)
print(best.seqB)

MRCIGM-SNRDFVEGVSGGSWVDIVLEHGSCVTTMAKNKPTLDFELIKTEAKQPATLRKYCIEAKLTNTTTESRCPTQGEPSLNEEQDKRFVCKHSMVDRGWGNGCGLFGKGGIVTCAMFRCKKNMEGKVVQPENLEYTIVITPHSGEEHAVGNDTGKHGKEIKITPQSSITEAELTGYGTVTMECSPRTGLDFNEMVLLQMENKAWLVHRQWFLDLPLPWLPGADTQGSNWIQKETLVTFKNPHAKKQDVVVLGSQEGAMHTALTGATEIQMSSGNLLFTGHLKCRLRMDKLQLKGMSYSMCTGKFKVVKEIAETQHGTIVIRVQYEGDGSPCKIPFEIMDLEKRHVLGRLITVNPIVTEKDSPVNIEAEPPFGDSYIIIGVEPGQLKLNWFKKGSSIGQMFETTMRGAKRMAILGDTAWDFGSLGGVFTSIGKALHQVFGAIYGAAFSGVSWTMKILIGVIITWIGMNSRSTSLSVTLVLVGIVTLYLGVMVQA
MRCIG-ISNRDFVEGVSGGSWVDIVLEHGSCVTTMAKNKPTLDFELIKTEAKQPATLRKYCIEAKLTNTTTESRCPTQGEPSLNEEQDKRFVCKHSMVDRGWGNGCGLFGKGGIVTCAMFRCKKNMEGKVVQPENLEYTIVITPHSGEEHAVGNDTGKHGKEIKITPQSSITEAELTGYGTVTMECSPRTGLDFNEMVLLQMENKAWLVHRQWFLDLPLPWLPGADTQGSNWIQKETLVTFKNPHAKKQDVVVLGSQEGAMHTALTGATEIQMSSGNLLFTGHLKCRLRMDKLQLKGMSYSMCTGKFKVVKEIAETQHGTIVIRVQYEGDGSPCKIPFEIMDLEKRHVLGRLITVNPIVTEKDSPVNIEAEPPFGDSYIIIGVEPGQLKLNWFKKGSSIGQMFETTMRGAKRMAILGDTAWDFGSLGGVFTSIGKALHQVFGAIYGAAFSGVSWTMKILIGVIITWIGMNSRSTSLSVTLVLVGIVTLYLGVMVQA


C:\Users\Sagor\AppData\Roaming\Python\Python312\site-packages\Bio\pairwise2.py:278: BiopythonDeprecationWarning: Bio.pairwise2 has been deprecated, and we intend to remove it in a future release of Biopython. As an alternative, please consider using Bio.Align.PairwiseAligner as a replacement, and contact the Biopython developers if you still need the Bio.pairwise2 module.
  warnings.warn(


In [3]:
mutations = []
pos = 0  # protein position (1-based)

for a, b in zip(best.seqA, best.seqB):
    if a != "-":
        pos += 1
    if a != b and a != "-" and b != "-":
        mutations.append(f"{a}{pos}{b}")

print(mutations)


[]


In [53]:
from Bio import SeqIO

# 1. Load your genome
record = SeqIO.read("MN328061_BD_2019_strain.fasta", "fasta")
genome_dna = record.seq

# 2. Translate the whole genome in the correct frame
# DENV is one long ORF, so we translate the whole thing
full_prot = str(genome_dna.translate(stop_symbol="?"))

# 3. Search for the "MRCIG" anchor (Universal for DENV-2 E-protein)
anchor = "MRCIGISNR"
start_pos = full_prot.find(anchor)

if start_pos != -1:
    # Extract the standard 495 amino acids of the E-protein
    clean_e_protein = full_prot[start_pos : start_pos + 495]
    
    # Map back to DNA for your records
    nt_start = start_pos * 3
    nt_end = (start_pos + 495) * 3
    
    print(f"Success! Found E-protein at Protein Index: {start_pos}")
    print(f"Nucleotide range: {nt_start} to {nt_end}")
    print(f"Resulting Sequence: {clean_e_protein[:30]}...")
    print(f"Final Length: {len(clean_e_protein)} aa")
else:
    print("Anchor not found. Checking alternate frame...")
    # This happens if the genome starts with 1 or 2 extra bases
    for shift in [1, 2]:
        prot = str(genome_dna[shift:].translate(stop_symbol="?"))
        if anchor in prot:
            print(f"Found it with a {shift} base shift!")

Anchor not found. Checking alternate frame...
Found it with a 1 base shift!


In [49]:
def get_clean_e_protein(fasta_path):
    record = SeqIO.read(fasta_path, "fasta")
    # Translate the whole thing, ignoring stop codons for the search
    full_prot = str(record.seq.translate(stop_symbol="?"))
    anchor = "MRCIGISNR"
    start_pos = full_prot.find(anchor)
    
    if start_pos != -1:
        # Extract the standard 495aa E-protein
        return full_prot[start_pos : start_pos + 495]
    return None

# Generate your pair for fine-tuning
seq_2019 = get_clean_e_protein("MN328061_BD_2019_strain.fasta")
seq_2023 = get_clean_e_protein("PQ657766_BD_2023_strain.fasta")

# Now you can print the differences directly
for i, (a, b) in enumerate(zip(seq_2019, seq_2023)):
    if a != b:
        print(f"Mutation found: {a}{i+1}{b}")

TypeError: 'NoneType' object is not iterable

In [51]:
def get_clean_e_protein(fasta_path):
    record = SeqIO.read(fasta_path, "fasta")
    genome_dna = record.seq
    anchor = "MRCIGISNR"
    
    # Try all 3 reading frames (0, 1, and 2 base offsets)
    for shift in [0, 1, 2]:
        # Translate the genome starting from the current shift
        full_prot = str(genome_dna[shift:].translate(stop_symbol="?"))
        start_pos = full_prot.find(anchor)
        
        if start_pos != -1:
            print(f"Found anchor in {fasta_path} with a {shift} base shift.")
            # Extract exactly 495 amino acids for the E-protein
            return full_prot[start_pos : start_pos + 495]
    
    print(f"Error: Anchor not found in {fasta_path} in any frame.")
    return None

# --- Re-run the extraction ---
seq_2019 = get_clean_e_protein("MN328061_BD_2019_strain.fasta")
seq_2023 = get_clean_e_protein("PQ657766_BD_2023_strain.fasta")

# --- Safety check before the loop ---
if seq_2019 and seq_2023:
    print("Both sequences extracted successfully. Comparing mutations...")
    for i, (a, b) in enumerate(zip(seq_2019, seq_2023)):
        if a != b:
            print(f"Mutation found: {a}{i+1}{b}")
else:
    print("One or both sequences are missing. Check your FASTA files.")

Found anchor in MN328061_BD_2019_strain.fasta with a 1 base shift.
Found anchor in PQ657766_BD_2023_strain.fasta with a 0 base shift.
Both sequences extracted successfully. Comparing mutations...
Mutation found: H52Q
Mutation found: A71E
Mutation found: T120R
Mutation found: I129V
Mutation found: N149H
Mutation found: I226T
Mutation found: S390N
Mutation found: V462I
Mutation found: S478T
Mutation found: V484I


In [54]:
from Bio.PDB import PDBParser
from Bio.SeqUtils import seq1

parser = PDBParser(QUIET=True)
structure = parser.get_structure("8FE3", "8FE3.pdb")

chain = structure[0]['A']   # usually chain A (we will verify)
pdb_seq = ""

pdb_res_ids = []

for res in chain:
    if res.id[0] == " ":  # standard amino acid
        pdb_seq += seq1(res.resname)
        pdb_res_ids.append(res.id[1])  # actual PDB residue number

print(len(pdb_seq))


386


In [55]:
from Bio import pairwise2

alignments = pairwise2.align.globalxx(
    str(seq_2023),   # your E_PQ657766 sequence
    pdb_seq
)

best = alignments[0]
seqA, seqB = best.seqA, best.seqB


In [56]:
mapping = {}

e_pos = 0
pdb_pos = 0

for a, b in zip(seqA, seqB):
    if a != "-":
        e_pos += 1
    if b != "-":
        pdb_pos += 1

    if a != "-" and b != "-":
        mapping[e_pos] = pdb_res_ids[pdb_pos - 1]


In [57]:
mapping[149]


149

In [58]:
mutations = [52, 71, 120, 129, 149, 226, 390, 462, 478, 484]

for m in mutations:
    if m in mapping:
        print(f"E{m} → PDB residue {mapping[m]}")
    else:
        print(f"E{m} not present in PDB (missing region)")


E52 → PDB residue 52
E71 → PDB residue 71
E120 → PDB residue 120
E129 → PDB residue 129
E149 → PDB residue 149
E226 → PDB residue 226
E390 → PDB residue 390
E462 not present in PDB (missing region)
E478 not present in PDB (missing region)
E484 not present in PDB (missing region)


mutation            domain
H52Q                Domain I
A71E	            Domain I	
T120R	            Domain I	
I129V	            Domain I	
N149H	            Domain I/II hinge	
I226T	            Domain II	
S390N	            Domain III
V462I	            TM	
S478T	            TM	
V484I	            TM	

In [7]:
from Bio import SeqIO, pairwise2
from Bio.PDB import PDBParser
from Bio.SeqUtils import seq1

# =========================
# 1. Load E protein sequence (2023 strain)
# =========================
record = SeqIO.read("E_PQ657766.fasta", "fasta")
e_seq = str(record.seq)

# =========================
# 2. Extract sequence from 8FE3 PDB
# =========================
parser = PDBParser(QUIET=True)
structure = parser.get_structure("8FE3", "8FE3.pdb")

chain = structure[0]['A']  # adjust if needed
pdb_seq = ""
pdb_res_ids = []

for res in chain:
    if res.id[0] == " ":
        pdb_seq += seq1(res.resname)
        pdb_res_ids.append(res.id[1])

# =========================
# 3. Align E protein to PDB sequence
# =========================
alignment = pairwise2.align.globalxx(e_seq, pdb_seq)[0]
seqA, seqB = alignment.seqA, alignment.seqB

# =========================
# 4. Build mapping_dict (E position → PDB residue number)
# =========================
mapping_dict = {}
e_pos = 0
pdb_pos = 0

for a, b in zip(seqA, seqB):
    if a != "-":
        e_pos += 1
    if b != "-":
        pdb_pos += 1
    if a != "-" and b != "-":
        mapping_dict[e_pos] = pdb_res_ids[pdb_pos - 1]

C:\Users\Sagor\AppData\Roaming\Python\Python312\site-packages\Bio\pairwise2.py:278: BiopythonDeprecationWarning: Bio.pairwise2 has been deprecated, and we intend to remove it in a future release of Biopython. As an alternative, please consider using Bio.Align.PairwiseAligner as a replacement, and contact the Biopython developers if you still need the Bio.pairwise2 module.
  warnings.warn(


In [8]:
def get_domain(position):
    if 1 <= position <= 150:
        return "Domain I"
    elif 151 <= position <= 300:
        return "Domain II"
    elif 301 <= position <= 395:
        return "Domain III"
    elif 396 <= position <= 495:
        return "Stem / Transmembrane"
    else:
        return "Unknown"

In [9]:
def is_resolved(position, mapping_dict):
    return position in mapping_dict

In [10]:
aa_property = {
    "A": "hydrophobic",
    "V": "hydrophobic",
    "I": "hydrophobic",
    "L": "hydrophobic",
    "M": "hydrophobic",
    "F": "hydrophobic",
    "W": "hydrophobic",
    "P": "hydrophobic",

    "G": "neutral",
    "S": "polar",
    "T": "polar",
    "N": "polar",
    "Q": "polar",
    "Y": "polar",
    "C": "polar",

    "D": "negatively charged",
    "E": "negatively charged",

    "K": "positively charged",
    "R": "positively charged",
    "H": "positively charged"
}

In [11]:
def get_chemical_change(ref, mut):
    return f"{aa_property[ref]} → {aa_property[mut]}"

In [13]:
def classify_mutation(ref, mut):
    if aa_property[ref] == aa_property[mut]:
        return "conservative"
    else:
        return "non-conservative"

In [21]:
def annotate_mutation(mutation):
    ref = mutation[0]
    pos = int(mutation[1:-1])
    mut = mutation[-1]

    return {
        "mutation": mutation,
        "position": pos,
        "domain": get_domain(pos),
        "pdb_residue": mapping_dict.get(pos, None),
        "resolved_in_pdb": pos in mapping_dict,
        "chemical_change": get_chemical_change(ref, mut),
        "mutation_type": classify_mutation(ref, mut)
    }

In [23]:
mutations = [
    "H52Q", "A71E", "T120R", "I129V", "N149H",
    "I226T", "S390N", "V462I", "S478T", "V484I"
]

annotations = [annotate_mutation(m) for m in mutations]

for a in annotations:
    print(a)

{'mutation': 'H52Q', 'position': 52, 'domain': 'Domain I', 'pdb_residue': 52, 'resolved_in_pdb': True, 'chemical_change': 'positively charged → polar', 'mutation_type': 'non-conservative'}
{'mutation': 'A71E', 'position': 71, 'domain': 'Domain I', 'pdb_residue': 71, 'resolved_in_pdb': True, 'chemical_change': 'hydrophobic → negatively charged', 'mutation_type': 'non-conservative'}
{'mutation': 'T120R', 'position': 120, 'domain': 'Domain I', 'pdb_residue': 120, 'resolved_in_pdb': True, 'chemical_change': 'polar → positively charged', 'mutation_type': 'non-conservative'}
{'mutation': 'I129V', 'position': 129, 'domain': 'Domain I', 'pdb_residue': 129, 'resolved_in_pdb': True, 'chemical_change': 'hydrophobic → hydrophobic', 'mutation_type': 'conservative'}
{'mutation': 'N149H', 'position': 149, 'domain': 'Domain I', 'pdb_residue': 149, 'resolved_in_pdb': True, 'chemical_change': 'polar → positively charged', 'mutation_type': 'non-conservative'}
{'mutation': 'I226T', 'position': 226, 'domai

In [24]:
import json
import random

def generate_sample(annotation):
    mutation = annotation["mutation"]
    domain = annotation["domain"]
    resolved = annotation["resolved_in_pdb"]
    chem = annotation["chemical_change"]
    mut_type = annotation["mutation_type"]
    pdb_res = annotation["pdb_residue"]

    instruction = f"Analyze the structural implication of mutation {mutation} in the Dengue virus type 2 Envelope protein."

    input_text = f"PDB Structure: 8FE3. Domain: {domain}. Position: {annotation['position']}."

    if resolved:
        output = (
            f"The {mutation} mutation occurs in {domain} of the Envelope protein. "
            f"This substitution changes the residue chemistry ({chem}) and is classified as a {mut_type} mutation. "
            f"Because residue {pdb_res} is resolved in the 8FE3 structure, this alteration may influence local structural interactions while maintaining the canonical fold."
        )
    else:
        output = (
            f"The {mutation} mutation is located in the {domain} region of the Envelope protein. "
            f"However, this position is not resolved in the 8FE3 structure, limiting direct structural interpretation. "
            f"The chemical change ({chem}) suggests potential effects on membrane-associated regions."
        )

    return {
        "instruction": instruction,
        "input": input_text,
        "output": output
    }


# Generate dataset
dataset = []
for ann in annotations:
    for _ in range(5):  # 5 variations per mutation (can increase)
        dataset.append(generate_sample(ann))

# Save to JSON
with open("prot2chat_tropical_dataset.json", "w") as f:
    json.dump(dataset, f, indent=4)

print("Dataset generated:", len(dataset), "samples")

Dataset generated: 50 samples


In [25]:
def generate_comparative_sample(annotations):
    mut_list = [a["mutation"] for a in annotations]

    instruction = "Compare the structural differences in the Envelope protein between the 2019 and 2023 Bangladesh strains."

    input_text = "PDB Structure: 8FE3. Strain comparison: Bangladesh 2019 vs 2023."

    output = (
        f"Comparison of the Envelope protein sequences reveals the following substitutions: {', '.join(mut_list)}. "
        "Most mutations are located within Domain I, with additional substitutions in Domains II and III. "
        "These changes occur primarily within structurally resolved regions, while several substitutions "
        "are located in the transmembrane region not present in the 8FE3 structure."
    )

    return {
        "instruction": instruction,
        "input": input_text,
        "output": output
    }

In [26]:
def generate_domain_summary(annotations):
    domain_counts = {}

    for a in annotations:
        d = a["domain"]
        domain_counts[d] = domain_counts.get(d, 0) + 1

    instruction = "Explain the domain-level distribution of mutations in the Dengue virus Envelope protein."

    input_text = "Structure: 8FE3."

    output = (
        "Mutation distribution analysis shows: " +
        ", ".join([f"{count} mutations in {domain}" for domain, count in domain_counts.items()]) +
        ". Domain I contains the highest concentration of substitutions, suggesting localized evolutionary variation."
    )

    return {
        "instruction": instruction,
        "input": input_text,
        "output": output
    }

In [27]:
def generate_absence_sample(annotation):
    if annotation["resolved_in_pdb"]:
        return None

    mutation = annotation["mutation"]
    domain = annotation["domain"]

    instruction = f"Why cannot mutation {mutation} be structurally analyzed using the 8FE3 model?"

    input_text = "PDB Structure: 8FE3."

    output = (
        f"The {mutation} substitution is located in the {domain} region of the Envelope protein. "
        "This region corresponds to the stem or transmembrane segment, which is not included "
        "in the resolved 8FE3 ectodomain structure. Therefore, direct structural interpretation is limited."
    )

    return {
        "instruction": instruction,
        "input": input_text,
        "output": output
    }

In [28]:
def generate_conservation_sample(annotations):
    instruction = "What does the preservation of the Envelope protein fold imply despite multiple amino-acid substitutions?"

    input_text = "Structure: 8FE3. Comparative strains: Bangladesh 2019 vs 2023."

    output = (
        "Despite multiple amino-acid substitutions, the Envelope protein retains its canonical three-domain fold. "
        "This suggests structural constraints that preserve functional integrity, particularly in receptor-binding "
        "and membrane-fusion regions."
    )

    return {
        "instruction": instruction,
        "input": input_text,
        "output": output
    }

In [29]:
dataset = []

# Pattern 1 (mutation-level)
for ann in annotations:
    dataset.append(generate_sample(ann))

# Pattern 2
dataset.append(generate_comparative_sample(annotations))

# Pattern 3
dataset.append(generate_domain_summary(annotations))

# Pattern 4
for ann in annotations:
    sample = generate_absence_sample(ann)
    if sample:
        dataset.append(sample)

# Pattern 5
dataset.append(generate_conservation_sample(annotations))

print("Total samples:", len(dataset))

Total samples: 16


In [30]:
mutation_question_templates = [
    "Analyze the structural implication of mutation {mutation}.",
    "What structural effects might arise from substitution {mutation}?",
    "Describe the impact of mutation {mutation} on the Envelope protein.",
    "Explain how {mutation} may alter structural interactions.",
    "Discuss the residue-level consequences of {mutation}."
]

In [31]:
mutation_output_templates = [
    "The {mutation} mutation occurs in {domain}. It represents a {mutation_type} substitution ({chemical_change}). Since this residue is {resolved_text}, it may influence local structural stability.",
    "Mutation {mutation} is located in {domain}. The chemical shift ({chemical_change}) suggests possible changes in residue interactions.",
    "{mutation} introduces a {mutation_type} alteration within {domain}, potentially affecting local conformation."
]

In [32]:
import random

def generate_scaled_mutation_sample(annotation):
    mutation = annotation["mutation"]
    domain = annotation["domain"]
    resolved = annotation["resolved_in_pdb"]
    chem = annotation["chemical_change"]
    mut_type = annotation["mutation_type"]

    resolved_text = "structurally resolved in 8FE3" if resolved else "not resolved in the 8FE3 structure"

    instruction = random.choice(mutation_question_templates).format(mutation=mutation)

    output_template = random.choice(mutation_output_templates)

    output = output_template.format(
        mutation=mutation,
        domain=domain,
        mutation_type=mut_type,
        chemical_change=chem,
        resolved_text=resolved_text
    )

    return {
        "instruction": instruction,
        "input": "PDB Structure: 8FE3",
        "output": output
    }

In [33]:
dataset = []

for ann in annotations:
    for _ in range(20):   # 20 variations per mutation
        dataset.append(generate_scaled_mutation_sample(ann))

# Add one instance of the other patterns (can scale similarly)
dataset.append(generate_comparative_sample(annotations))
dataset.append(generate_domain_summary(annotations))
dataset.append(generate_conservation_sample(annotations))

for ann in annotations:
    sample = generate_absence_sample(ann)
    if sample:
        dataset.append(sample)

print("Total samples:", len(dataset))

Total samples: 206


In [34]:
import json

with open("prot2chat_tropical_dataset.json", "w") as f:
    json.dump(dataset, f, indent=4)

print("Dataset saved.")

Dataset saved.


In [35]:
# comparative question

comparative_question_templates = [
    "Compare the structural differences in the Envelope protein between the 2019 and 2023 Bangladesh strains.",
    "What residue-level variations distinguish the 2019 and 2023 DENV-2 E proteins?",
    "How do the Envelope proteins differ structurally between the two Bangladesh isolates?",
    "Describe the mutation pattern observed between the 2019 and 2023 strains.",
    "Summarize the structural changes identified in the comparative strain analysis."
]

comparative_output_templates = [
    "Comparison of the Envelope protein sequences reveals substitutions including {mut_list}. Most mutations are localized in Domain I, with additional substitutions in Domains II and III. Several changes are resolved in the 8FE3 structure.",
    "The two strains differ by the following mutations: {mut_list}. These substitutions are primarily clustered within the ectodomain, while some occur in unresolved transmembrane regions.",
    "Sequence comparison identifies multiple amino-acid substitutions ({mut_list}) that map largely to structurally resolved regions of the Envelope protein."
]

def generate_scaled_comparative_sample(annotations):
    mut_list = ", ".join([a["mutation"] for a in annotations])
    
    instruction = random.choice(comparative_question_templates)
    output_template = random.choice(comparative_output_templates)

    output = output_template.format(mut_list=mut_list)

    return {
        "instruction": instruction,
        "input": "PDB Structure: 8FE3. Comparative analysis.",
        "output": output
    }

In [36]:
domain_question_templates = [
    "Explain the domain-level distribution of mutations in the Envelope protein.",
    "How are mutations distributed across Domains I, II, and III?",
    "Describe the clustering pattern of substitutions within structural domains.",
    "Which structural domains contain the highest mutation density?",
    "Analyze the domain-wise mutation frequency."
]

domain_output_templates = [
    "Mutation analysis shows the following domain distribution: {domain_summary}. Domain I contains the highest concentration of substitutions.",
    "Substitutions are distributed as follows: {domain_summary}. This suggests localized variation within specific structural regions.",
    "Domain-level mapping indicates: {domain_summary}, highlighting non-uniform mutation clustering."
]

def generate_scaled_domain_sample(annotations):
    domain_counts = {}
    for a in annotations:
        d = a["domain"]
        domain_counts[d] = domain_counts.get(d, 0) + 1

    domain_summary = ", ".join([f"{count} in {domain}" for domain, count in domain_counts.items()])

    instruction = random.choice(domain_question_templates)
    output_template = random.choice(domain_output_templates)

    output = output_template.format(domain_summary=domain_summary)

    return {
        "instruction": instruction,
        "input": "PDB Structure: 8FE3",
        "output": output
    }

In [37]:
absence_question_templates = [
    "Why cannot mutation {mutation} be structurally analyzed using 8FE3?",
    "Why is {mutation} not visible in the 8FE3 structure?",
    "Explain the structural limitation for analyzing {mutation}.",
    "Why does the 8FE3 model not resolve mutation {mutation}?",
    "Discuss why {mutation} lacks structural representation."
]

absence_output_templates = [
    "The {mutation} substitution occurs within the {domain} region, which corresponds to the stem or transmembrane segment not included in the 8FE3 ectodomain structure.",
    "Mutation {mutation} lies in the {domain} region, absent from the resolved cryo-EM model, preventing direct structural interpretation.",
    "{mutation} is located outside the resolved portion of the 8FE3 structure, limiting residue-level analysis."
]

def generate_scaled_absence_sample(annotation):
    if annotation["resolved_in_pdb"]:
        return None

    mutation = annotation["mutation"]
    domain = annotation["domain"]

    instruction = random.choice(absence_question_templates).format(mutation=mutation)
    output_template = random.choice(absence_output_templates)

    output = output_template.format(mutation=mutation, domain=domain)

    return {
        "instruction": instruction,
        "input": "PDB Structure: 8FE3",
        "output": output
    }

In [38]:
conservation_question_templates = [
    "What does preservation of the Envelope fold imply despite substitutions?",
    "How can the E protein maintain structural stability despite mutations?",
    "Why does the Envelope protein retain its canonical fold?",
    "What structural constraints preserve the E protein architecture?",
    "Discuss structural conservation despite sequence variation."
]

conservation_output_templates = [
    "Despite multiple amino-acid substitutions, the Envelope protein maintains its canonical three-domain architecture, indicating structural constraints essential for viral function.",
    "The persistence of the canonical fold suggests that mutations occur without disrupting global structural integrity.",
    "Structural conservation implies evolutionary pressure to preserve receptor-binding and membrane-fusion functionality."
]

def generate_scaled_conservation_sample():
    instruction = random.choice(conservation_question_templates)
    output = random.choice(conservation_output_templates)

    return {
        "instruction": instruction,
        "input": "PDB Structure: 8FE3",
        "output": output
    }

In [39]:
dataset = []

# Pattern 1 (already scaled mutation samples)
for ann in annotations:
    for _ in range(30):  # adjust scaling
        dataset.append(generate_scaled_mutation_sample(ann))

# Pattern 2
for _ in range(20):
    dataset.append(generate_scaled_comparative_sample(annotations))

# Pattern 3
for _ in range(20):
    dataset.append(generate_scaled_domain_sample(annotations))

# Pattern 4
for ann in annotations:
    for _ in range(10):
        sample = generate_scaled_absence_sample(ann)
        if sample:
            dataset.append(sample)

# Pattern 5
for _ in range(20):
    dataset.append(generate_scaled_conservation_sample())

print("Total samples:", len(dataset))

Total samples: 390


In [40]:
import json

with open("prot2chat_tropical_dataset.json", "w") as f:
    json.dump(dataset, f, indent=4)

print("Dataset saved.")

Dataset saved.


In [ ]:
import json

dataset = []

# -------------------------
#  Mutation → Structure
# -------------------------

for ann in annotations:
    mutation = ann["mutation"]
    domain = ann["domain"]
    resolved = ann["resolved_in_pdb"]
    chem = ann["chemical_change"]
    mut_type = ann["mutation_type"]

    resolved_text = (
        "structurally resolved in 8FE3"
        if resolved else
        "not resolved in the 8FE3 structure"
    )

    for q_template in mutation_question_templates:
        for o_template in mutation_output_templates:

            instruction = q_template.format(mutation=mutation)

            output = o_template.format(
                mutation=mutation,
                domain=domain,
                mutation_type=mut_type,
                chemical_change=chem,
                resolved_text=resolved_text
            )

            dataset.append({
                "instruction": instruction,
                "input": "PDB Structure: 8FE3",
                "output": output
            })


# -------------------------
#  Comparative Reasoning
# -------------------------

mut_list = ", ".join([a["mutation"] for a in annotations])

for q_template in comparative_question_templates:
    for o_template in comparative_output_templates:

        dataset.append({
            "instruction": q_template,
            "input": "PDB Structure: 8FE3. Comparative strain analysis.",
            "output": o_template.format(mut_list=mut_list)
        })


# -------------------------
#  Domain Distribution
# -------------------------

domain_counts = {}
for a in annotations:
    d = a["domain"]
    domain_counts[d] = domain_counts.get(d, 0) + 1

domain_summary = ", ".join(
    [f"{count} in {domain}" for domain, count in domain_counts.items()]
)

for q_template in domain_question_templates:
    for o_template in domain_output_templates:

        dataset.append({
            "instruction": q_template,
            "input": "PDB Structure: 8FE3",
            "output": o_template.format(domain_summary=domain_summary)
        })


# -------------------------
# 4️⃣ Absence-Aware
# -------------------------

for ann in annotations:
    if not ann["resolved_in_pdb"]:

        mutation = ann["mutation"]
        domain = ann["domain"]

        for q_template in absence_question_templates:
            for o_template in absence_output_templates:

                dataset.append({
                    "instruction": q_template.format(mutation=mutation),
                    "input": "PDB Structure: 8FE3",
                    "output": o_template.format(
                        mutation=mutation,
                        domain=domain
                    )
                })


# -------------------------
# 5️⃣ Conservation Reasoning
# -------------------------

for q_template in conservation_question_templates:
    for o_template in conservation_output_templates:

        dataset.append({
            "instruction": q_template,
            "input": "PDB Structure: 8FE3",
            "output": o_template
        })


# -------------------------
# 🔹 Remove Exact Duplicates
# -------------------------

unique_dataset = []
seen = set()

for sample in dataset:
    key = (sample["instruction"], sample["input"], sample["output"])
    if key not in seen:
        seen.add(key)
        unique_dataset.append(sample)

dataset = unique_dataset


# -------------------------
# 🔹 Save JSON
# -------------------------

with open("prot2chat_tropical_dataset2.json", "w") as f:
    json.dump(dataset, f, indent=4)

print("Final dataset size:", len(dataset))

Final dataset size: 240
